In [1]:
import anndata as ad
import numpy as np
import pandas as pd
import gcsfs
import pyarrow.dataset as ds
import glob

In [2]:
pd.set_option("display.max_columns", 50)

### tahoe

In [4]:
'''
fs = gcsfs.GCSFileSystem()
gcp_base_path = "gs://arc-institute-virtual-cell-atlas/tahoe100M/2025-02-25/"
infile = "/".join([gcp_base_path.rstrip("/"), 'metadata', 'obs_metadata.parquet'])
metadata = ds.dataset(infile, filesystem=fs, format="parquet").to_table().to_pandas()
metadata.to_parquet('obs_metadata.parquet', index=False)
'''

'\nfs = gcsfs.GCSFileSystem()\ngcp_base_path = "gs://arc-institute-virtual-cell-atlas/tahoe100M/2025-02-25/"\ninfile = "/".join([gcp_base_path.rstrip("/"), \'metadata\', \'obs_metadata.parquet\'])\nmetadata = ds.dataset(infile, filesystem=fs, format="parquet").to_table().to_pandas()\nmetadata.to_parquet(\'obs_metadata.parquet\', index=False)\n'

In [5]:
#read original data
metadata = pd.read_parquet('../../obs_metadata.parquet')
tahoe_drugs = metadata.drop_duplicates('drug')[['drug']]

In [6]:
#read lamin data
tahoe_raw = pd.concat(
    [pd.read_parquet(f) for f in sorted(glob.glob('../../../data/tahoe/raw/plate*_obs.parquet'))],
    ignore_index=True
)

tahoe_raw = tahoe_raw[['pert_compound', 'pert_name']].drop_duplicates().reset_index(drop=True).astype(str)

In [7]:
#read our data
tahoe = ad.read_h5ad('../../../data/tahoe/pseudobulk_processed/sep_rep/tahoe_processed.h5ad')
tahoe_obs = tahoe.obs
tahoe_sm = tahoe_obs.drop_duplicates(['perturbagen', 'pubchem_cid'])[['perturbagen', 'pubchem_cid']].reset_index(drop=True)

In [8]:
df_tahoe = tahoe_sm.merge(tahoe_raw, left_on='perturbagen', right_on='pert_compound', how='left').merge(tahoe_drugs, left_on='pert_name', right_on='drug', how='left')

In [9]:
df_tahoe[df_tahoe['perturbagen'] != df_tahoe['pert_compound']]

,perturbagen,pubchem_cid,pert_compound,pert_name,drug


In [10]:
df_tahoe[df_tahoe['pert_name'] != df_tahoe['drug']]

,perturbagen,pubchem_cid,pert_compound,pert_name,drug


In [11]:
df_tahoe = df_tahoe[['perturbagen', 'pert_name']].rename(columns={'pert_name': 'original_pert_name'})

In [12]:
df_tahoe['dataset'] = 'tahoe'

## sciplex

In [13]:
#downloaded from https://zenodo.org/records/7041849/files/SrivatsanTrapnell2020_sciplex3.h5ad?download=1
#read original data
sci_orig = ad.read_h5ad('../../../sciplex/SrivatsanTrapnell2020_sciplex3.h5ad')
sci_drugs = sci_orig.obs.drop_duplicates('perturbation')[['perturbation']].reset_index(drop=True)
sci_drugs['perturbation'] = sci_drugs['perturbation'].astype(str)

In [14]:
#read lamin data
sci_raw = pd.read_parquet('../../../data/sciplex/raw/srivatsan20_sciplex3_obs.parquet')

In [15]:
sci_raw = sci_raw[['pert_compound', 'pert_name']].drop_duplicates().reset_index(drop=True).astype(str)

In [16]:
sci_raw.loc[sci_raw['pert_compound'] == 'control', 'pert_compound'] = 'DMSO'

In [17]:
#read our data
sci = ad.read_h5ad('../../../data/sciplex/pseudobulk_processed/sep_rep/srivatsan20_sciplex3_processed.h5ad')
sci_obs = sci.obs
sci_sm = sci_obs.drop_duplicates(['perturbagen', 'pubchem_cid'])[['perturbagen', 'pubchem_cid']].reset_index(drop=True)

In [18]:
df_sci = sci_sm.merge(sci_raw, left_on='perturbagen', right_on='pert_compound', how='left').merge(sci_drugs, left_on='pert_name', right_on='perturbation', how='left')

In [19]:
df_sci[df_sci['perturbagen'] != df_sci['pert_compound']]

,perturbagen,pubchem_cid,pert_compound,pert_name,perturbation


In [20]:
df_sci[df_sci['pert_name'] != df_sci['perturbation']]

,perturbagen,pubchem_cid,pert_compound,pert_name,perturbation


In [21]:
df_sci = df_sci[['perturbagen', 'pert_name']].rename(columns={'pert_name': 'original_pert_name'})

In [22]:
df_sci['dataset'] = 'sciplex3'

In [23]:
df_fixed_names = pd.concat([df_tahoe, df_sci])

In [24]:
df_fixed_names

,perturbagen,original_pert_name,dataset
0,alpelisib,Alpelisib,tahoe
1,Lapatinib ditosylate,Lapatinib ditosylate,tahoe
2,palbociclib,palbociclib,tahoe
3,Infigratinib,Infigratinib,tahoe
4,Erdafitinib,Erdafitinib,tahoe
...,...,...,...
184,BMS-265246,BMS-265246,sciplex3
185,CUDC-101,CUDC-101,sciplex3
186,JNJ-26854165,JNJ-26854165 (Serdemetan),sciplex3
187,curcumin,Curcumin,sciplex3


## fix names

In [25]:
df_compounds = pd.read_pickle('../../tahoe_sci_op3.pkl')

In [26]:
df = df_compounds.merge(df_fixed_names, on=['perturbagen', 'dataset'], how='left')

In [27]:
df['original_pert_name'] = df['original_pert_name'].fillna(df['perturbagen'])

## fix smiles

In [28]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import ConvertToNumpyArray

def smiles_to_fingerprints(smiles_list, radius=1, fp_size=2000):
    """Convert a list of SMILES strings to Morgan (ECFP) fingerprint matrix.

    Args:
        smiles_list: Iterable of SMILES strings.
        radius: Morgan fingerprint radius (default 1, i.e. ECFP2).
        fp_size: Fingerprint bit vector size.

    Returns:
        np.ndarray of shape (len(smiles_list), fp_size), dtype uint8.
        Invalid SMILES yield a zero vector for that row.
    """
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_size)
    fps = []
    for smile in smiles_list:
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            fps.append(None)
        else:
            fp = gen.GetFingerprint(mol)
            arr = np.zeros(fp_size, dtype=np.uint8)
            ConvertToNumpyArray(fp, arr)
            fps.append(arr)
    return fps

In [29]:
df = df.rename(columns={'ECPF:2': 'ECFP:2'})

In [30]:
df['ECFP:2'] = smiles_to_fingerprints(df['smiles'])

In [31]:
df[df['perturbagen'] == 'tanespimycin']

,perturbagen,pubchem_cid,smiles,dataset,cmap_name,symbol,code,symbol_,ECFP:2,LPM_emb,original_pert_name
498,tanespimycin,6505803,CC1CC(C(C(C=C(C(C(C=CC=C(C(=O)NC2=CC(=O)C(=C(C...,sciplex3,17-AAG,17-AAG-10uM,4.0,17-AAG,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.2399607002735138, -0.15076814591884613, -0...",Tanespimycin (17-AAG)
499,tanespimycin,6505803,CC1CC(C(C(C=C(C(C(C=CC=C(C(=O)NC2=CC(=O)C(=C(C...,sciplex3,tanespimycin,tanespimycin-10uM,10028.0,tanespimycin,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.018972985446453094, -0.24066485464572906, 0...",Tanespimycin (17-AAG)


In [97]:
#df.to_pickle("tahoe_sci_op3_updated.pkl")

In [ ]:
#tanespimycin

In [135]:
#pd.read_pickle("dili.pkl").rename(columns={'ECPF:2': 'ECFP:2'}).to_pickle("dili.pkl")
#pd.read_pickle("tahoe_sci_op3_updated.pkl").rename(columns={'ECPF:2': 'ECFP:2'}).to_pickle("tahoe_sci_op3_updated.pkl")